# Setup check · SciML for Computational Mechanics (NPTEL Masterclass)
Run this once before Session 1: **Runtime → Run all**. It takes under a minute and checks everything the worksheets need.

If the last cell prints **You are ready for the masterclass**, you are done. If not, send the output to the organisers before Session 1.

In [ ]:
import sys, time, platform
import numpy as np, torch, matplotlib
print(f"Python {platform.python_version()} · PyTorch {torch.__version__} · NumPy {np.__version__}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "(a CPU is fine for every worksheet)")
ok = {}
major, minor = (int(v) for v in torch.__version__.split(".")[:2])
ok["PyTorch 2.0 or newer"] = (major, minor) >= (2, 0)

In [ ]:
# Session 3 differentiates network outputs with torch.func
try:
    from torch.func import jacfwd, vmap
    f = lambda x: torch.sin(x)
    d2 = vmap(jacfwd(jacfwd(f)))(torch.linspace(0, 1, 5))
    ok["torch.func available"] = torch.allclose(d2, -torch.sin(torch.linspace(0, 1, 5)), atol=1e-6)
except Exception as e:
    print("torch.func problem:", e); ok["torch.func available"] = False

In [ ]:
# Session 1 needs second derivatives through autograd
x = torch.linspace(0, 1, 50).reshape(-1, 1).requires_grad_(True)
u = x**3
ux = torch.autograd.grad(u, x, torch.ones_like(u), create_graph=True)[0]
uxx = torch.autograd.grad(ux, x, torch.ones_like(ux))[0]
ok["autograd 2nd-derivative test"] = torch.allclose(uxx, 6 * x.detach(), atol=1e-5)

In [ ]:
# A tiny physics-informed network: solve u'' = -pi^2 sin(pi x), u(0) = u(1) = 0
torch.manual_seed(0)
net = torch.nn.Sequential(torch.nn.Linear(1, 32), torch.nn.Tanh(), torch.nn.Linear(32, 32), torch.nn.Tanh(), torch.nn.Linear(32, 1))
opt = torch.optim.Adam(net.parameters(), 1e-3)
t0 = time.time()
for it in range(500):
    xc = torch.rand(128, 1, requires_grad=True)
    uc = net(xc)
    ux = torch.autograd.grad(uc, xc, torch.ones_like(uc), create_graph=True)[0]
    uxx = torch.autograd.grad(ux, xc, torch.ones_like(ux), create_graph=True)[0]
    xb = torch.tensor([[0.0], [1.0]])
    loss = ((uxx + np.pi**2 * torch.sin(np.pi * xc)) ** 2).mean() + 10 * (net(xb) ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step()
elapsed = time.time() - t0
ok["tiny PINN trains (loss falls)"] = loss.item() < 5.0
print(f"500 iterations in {elapsed:.1f} s, final loss {loss.item():.2e}")

In [ ]:
import matplotlib.pyplot as plt
xe = torch.linspace(0, 1, 200).reshape(-1, 1)
with torch.no_grad(): plt.plot(xe, net(xe), label="tiny PINN (500 iterations)")
plt.plot(xe, torch.sin(np.pi * xe), "--", label="exact sin(πx)"); plt.legend(); plt.title("plots work"); plt.show()
ok["matplotlib plots"] = True

In [ ]:
for k, v in ok.items(): print(f"{k:32s} {'✓' if v else '✗'}")
print()
print("You are ready for the masterclass." if all(ok.values()) else "Something is missing: send this output to the organisers before Session 1.")